In [18]:
# NECESSARY IMPORTS
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_core.tools import FunctionTool
from autogen_agentchat.messages import AgentEvent, ChatMessage
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
import requests
import re
import io
import PyPDF2

In [19]:
# Create an instance of the OpenAIChatCompletionClient

# Using LM studio for accesing models
model_client = OpenAIChatCompletionClient(
    # Specify the model to use
    model="meta-llama-3.1-8b-instruct",
    # Set the base URL for the API
    base_url="http://localhost:1234/v1",
    # Provide the API key for authentication
    api_key='lm-studio',
    # Additional model information
    model_info={
        # Indicate whether the model supports vision tasks
        "vision": False,
        # Indicate whether the model supports function calling
        "function_calling": True,
        # Indicate whether the model outputs JSON
        "json_output": False,
        # Specify the model family
        "family": "unknown",
    },
)

In [20]:
topic = input('Enter the topic name for which you want to create literature review')

In [21]:
# List of research papers related to multi-agent systems using LLMs
multi_agent_llm_research_paper  = [
    {'title': 'LLM Multi-Agent Systems: Challenges and Open Problems' , 'url' : 'https://arxiv.org/pdf/2402.03578'},
    {'title': 'EVOAGENT: Towards Automatic Multi-Agent Generation via EvolutionaryAlgorithms' , 'url': 'https://arxiv.org/pdf/2406.14228'} ,
    {'title': 'PROMPT INFECTION: LLM-TO-LLM PROMPT INJEC-TION WITHIN MULTI-AGENT SYSTEMS' , 'url': 'https://arxiv.org/pdf/2410.07283'} ,
    {'title': 'AgentLite: A Lightweight Library for Building and AdvancingTask-Oriented LLM Agent System' , 'url': 'https://arxiv.org/pdf/2402.15538'} ,
    {'title': 'Can LLMs Understand Social Norms in Autonomous Driving Games?' , 'url': 'https://arxiv.org/pdf/2408.12680'} ,
    {'title': 'Adapting LLM Agents with Universal Feedback in Communication' , 'url': 'https://arxiv.org/pdf/2310.01444'} ,
]

# List of research papers related to prompt engineering in LLMs
prompt_research_paper = [
    {'title': 'Control Flow-Augmented Decompiler based on Large Language Model', 'url': 'https://arxiv.org/pdf/2503.07215'},
    {'title': 'Rethinking Prompt-based Debiasing in Large Language Models', 'url': 'https://arxiv.org/pdf/2503.09219'},
    {'title': 'Prompt2LVideos: Exploring Prompts for Understanding Long-Form Multimodal Videos', 'url': 'https://arxiv.org/pdf/2503.08335'},
    {'title': 'Modeling Variants of Prompts for Vision-Language Models', 'url': 'https://arxiv.org/pdf/2503.08229'},
    {'title': 'Instruction-Augmented Long-Horizon Planning: Embedding Grounding Mechanisms in Embodied Mobile Manipulation', 'url': 'https://arxiv.org/pdf/2503.08084'},
    {'title': 'Visual and Text Prompt Segmentation: A Novel Multi-Model Framework for Remote Sensing', 'url': 'https://arxiv.org/pdf/2503.07911'},
]
research_papers = []

In [22]:
# Check if the word 'prompt' is in the topic (case insensitive)
if 'prompt' in topic.lower():
    # If 'prompt' is found in the topic, assign the list of prompt-related research papers
    research_papers = prompt_research_paper
else:
    # If 'prompt' is not found in the topic, assign the list of multi-agent LLM research papers
    research_papers = multi_agent_llm_research_paper

In [23]:
# Function to download PDF and process from URL
def download_and_process_paper(url: str) -> str:
    """Download a paper and extract content from abstract to before references."""

    # Send a GET request to the URL with a timeout of 30 seconds
    response = requests.get(url, timeout=30)
    # Check if the response status code is not 200 (OK)
    if response.status_code != 200:
        return ""

    # Read the PDF content from the response
    pdf_file = io.BytesIO(response.content)
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    text = ""
    abstract_found = False
    in_references = False

    # Loop through each page in the PDF
    for page_num in range(len(pdf_reader.pages)):
        # Extract text from the current page
        page_text = pdf_reader.pages[page_num].extract_text() or ""
        page_text_lower = page_text.lower()

        # Check if the abstract section is found
        if not abstract_found and ("abstract" in page_text_lower or re.search(r'^\s*abstract\s*$', page_text_lower, re.MULTILINE)):
            abstract_found = True
            abstract_start = page_text_lower.find("abstract")
            text += page_text[abstract_start:] + "\n"
            continue

        # Check if the references section is found
        if abstract_found and not in_references:
            if "references" in page_text_lower or re.search(r'^\s*references\s*$', page_text_lower, re.MULTILINE):
                in_references = True
                ref_start = page_text_lower.find("references")
                text += page_text[:ref_start]
                break
            text += page_text + "\n"

    # Clean up the extracted text by removing extra whitespace
    text = re.sub(r'\s+', ' ', text.strip())
    # Return the extracted text or the first 100 characters of the first page if no text is found
    return text if text else pdf_reader.pages[0].extract_text()

# Arxiv search function
def arxiv_search_and_fetch_content(query: str, max_results: int = 6) -> list:
    """
    Search Arxiv for papers and return the results including abstracts.
    """

    results = []
    # Loop through each paper in the research_papers list
    for paper in research_papers:
        title = paper['title']
        url = paper['url']
        # Download and process the paper to extract content
        text = download_and_process_paper(url)
        # Append the title and content to the results list
        results.append({
            "title": title,
            "content": text,
        })

    return results

In [ ]:
# Create a tool for fetching and processing papers from Arxiv
arxiv_fetch_paper_and_process = FunctionTool(
    arxiv_search_and_fetch_content, 
    # Description of the tool's functionality
    description="Search Arxiv for papers related to a given topic, including abstracts and returns a dictionary of paper title and content"
)

# Create an agent for downloading and processing papers
download_and_process_agent = AssistantAgent(
    name="Download_and_Process_Agent",
    # Assign the tool to the agent
    tools=[arxiv_fetch_paper_and_process],
    # Use the model client for the agent
    model_client=model_client,
    # Description of the agent's specialization
    description=f"An agent which is specializing in {topic} that can download papers based on the urls given and process pdf content and returns content of multiple papers",
    # System message defining the agent's role and behavior
    system_message="""You are a Download and Process assistant specialized in fetching research papers based on url given from arxiv webpage, 
                    and then process the downloaded pdf. You return the content of every requested paper (through their URL) and send it to the next 
                    agent in a dictionary with key title and content""",
)

# Create an agent for summarizing papers
summary_agent = AssistantAgent(
    name="Summary_Agent",
    model_client=model_client,
    description=f"An agent which is specializing in {topic} that can read and summarize the content",
    system_message="""You are a summarizer assistant specialized in reading and summarizing academic papers. 
    When given the content of research papers, summarize its key points, methodology, 
    results, and conclusions in 200 words. Focus on extracting the most relevant information for the literature review topic.""",
)

# Create an agent for generating literature review
literature_review_agent = AssistantAgent(
    name="Literature_Review_Agent",
    model_client=model_client,
    description="Generate a literature review based on paper summaries",
    system_message=f"""You are a Literature review assistant specialized in the field {topic} who can write literature 
    reviews of scientific and research papers. Your task is to synthesize the summaries of 
    multiple papers into a coherent literature review of at max 500 words. Organize the review by themes
    or chronologically as appropriate. After presenting the review, ask the user for feedback and be prepared to revise based on their input""",
)

# Create an agent for processing user feedback
# Can command either summary agent or literature review agent based on user prompt
feedback_agent = AssistantAgent(
    name="Feedback_Agent",
    model_client=model_client,
    description="Process user feedback and suggest improvements to the literature review and summary agent",
    system_message=f"""You are a feedback assistant specialized in the field {topic} who can incorporate user feedback to improve literature reviews or summaries. When 
    given feedback on a literature review, user will mention in prompt about Literature review, then analyze it carefully and suggest specific improvements to the literature review agent. When user says to
    change summary by mentioned summarizer in prompt for cases when summary of some papers are missing, then send this to summary agent. By default, send the feedback to the literature review agent.
    Focus on addressing the user's concerns while maintaining the academic quality of the review. If the user is satisfied and no further improvements are 
    needed, acknowledge this and suggest finalizing the review.""",
)

# Create a user proxy agent - fixed based on the class definition
user_proxy = UserProxyAgent(
    name="User",
    description="You are a user proxy agent which provides feedback to the generated literature review by feedback agent."
)

# Create a termination condition based on text mention
termination = TextMentionTermination("TERMINATE")



In [25]:
from typing import Sequence

final_review = None

# Define a selector function to determine the next agent to handle the message
def selector_func_with_user_proxy(messages: Sequence[AgentEvent | ChatMessage]) -> str | None:
    
    # If the last message is from the download and process agent, select the summary agent
    if messages[-1].source == download_and_process_agent.name:
        return summary_agent.name
    
    # If the last message is from the summary agent, select the literature review agent
    if messages[-1].source == summary_agent.name:
        return literature_review_agent.name
    
    # If the last message is from the literature review agent, select the user proxy agent
    if messages[-1].source == literature_review_agent.name:
        return user_proxy.name
    
    # If the last message is from the feedback agent
    if messages[-1].source == feedback_agent.name:
        # If the previous message contains 'summarizer', select the summary agent
        if 'summarizer' in messages[-2].content.lower():
            return summary_agent.name
        # Otherwise, select the literature review agent
        return literature_review_agent.name
    
    # If the last message is from the user proxy agent
    if messages[-1].source == user_proxy.name:
        # If there are more than 2 messages and the last message contains 'APPROVE', terminate the process
        if len(messages) > 2:
            if "APPROVE" in messages[-1].content.upper():
                final_review = messages[-1].content 
                return None
        # Otherwise, select the feedback agent
        return feedback_agent.name
        
    # Default to the download and process agent
    return download_and_process_agent.name

# Prompt for the selector to start the task
selector_prompt = "Select Download and process agent to start task"

# Create a team with all agents and the selector function
team = SelectorGroupChat(
    [download_and_process_agent, summary_agent, literature_review_agent, feedback_agent, user_proxy],
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    selector_func=selector_func_with_user_proxy,
    allow_repeated_speaker=True,
)

# Run the team and await the console output
task = f"You have to implement literature review for the topic {topic}"
await Console(team.run_stream(task=task))


---------- user ----------
You have to implement literature review for the topic multi agent LLM
---------- Download_and_Process_Agent ----------
[FunctionCall(id='283358743', arguments='{"query":"multi agent LLM","max_results":"10"}', name='arxiv_search_and_fetch_content')]
---------- Download_and_Process_Agent ----------
[FunctionExecutionResult(content='[{\'title\': \'LLM Multi-Agent Systems: Challenges and Open Problems\', \'content\': \'Abstract This paper explores existing works of multi-agent systems and identify challenges that remain in- adequately addressed. By leveraging the diverse capabilities and roles of individual agents within a multi-agent system, these systems can tackle com- plex tasks through collaboration. We discuss opti- mizing task allocation, fostering robust reasoning through iterative debates, managing complex and layered context information, and enhancing mem- ory management to support the intricate interac- tions within multi-agent systems. We also explore

TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, content='You have to implement literature review for the topic multi agent LLM', type='TextMessage'), ToolCallRequestEvent(source='Download_and_Process_Agent', models_usage=RequestUsage(prompt_tokens=326, completion_tokens=31), metadata={}, content=[FunctionCall(id='283358743', arguments='{"query":"multi agent LLM","max_results":"10"}', name='arxiv_search_and_fetch_content')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(source='Download_and_Process_Agent', models_usage=None, metadata={}, content=[FunctionExecutionResult(content='[{\'title\': \'LLM Multi-Agent Systems: Challenges and Open Problems\', \'content\': \'Abstract This paper explores existing works of multi-agent systems and identify challenges that remain in- adequately addressed. By leveraging the diverse capabilities and roles of individual agents within a multi-agent system, these systems can tackle com- plex tasks through collaboratio